# SwarmSort Gen 2, notebook 1: Arm A (Ultralytics defaults)

**What:** full YOLOv8n training on the Gen 2 dataset with the Ultralytics default hyperparameters (lr0 0.01, lrf 0.01,
momentum 0.937, weight_decay 0.0005, mosaic 1.0, scale 0.5): 100 epochs, **imgsz 640**, batch 16, seed 42,
deterministic, SGD, warmup 3 epochs, mosaic closed for the last 10, one GPU. Validation runs every epoch; the reported
numbers are the val metrics of `best.pt`.

**Why:** Arm A is the reference the two searched arms (B: random search, C: PSO) must beat. It needs no search, so it
can run while the searches run.

**Input:** Add Input -> Notebook Output -> **`g2_0_build_dataset`** (its finished version). No dataset needs to be
attached. **Accelerator: GPU T4 x2**, **Internet: on** (for the clone, pip and the `yolov8n.pt` download).
Run it with **Save Version -> Save & Run All (Commit)**.

**Expected time: about 2.4 hours.** Gen 1's Arm A took 4263 s for 100 epochs on 7324 train images at imgsz 416:

- pixel ratio: (640 / 416)² = 409,600 / 173,056 = **2.367**
- train-image ratio: Gen 2 expects 3000 (Gen 1 subsample) + 0.7 x 1500 (TACO) + 0.7 x 3200 (HITL) = 3000 + 1050 + 2240 = **6290** train images; 6290 / 7324 = **0.859**
- 4263 s x 2.367 x 0.859 = **8,666 s = 144 min = 2.4 h**

The real train count is in the build report. If JPEG decoding on Kaggle's CPUs limits
the speed on Kaggle's 4 CPU cores, allow up to about 30% more (about 3.1 h). Either way it is well inside the 12-hour limit.

**Freeze protection.** The attached dataset is first copied to local disk (a few minutes; the time is printed), because two Gen 2 runs froze about 55 minutes in while reading it from the `/kaggle/input` mount. Every script runs under a watchdog (`src/watchdog_run.py`): after 20 minutes without output or file changes it is killed with its dataloader workers and restarted, at most 3 times, and the restart continues the work. **If a version stopped or froze, attach its output and run again: it continues.**

Outputs: `/kaggle/working/SwarmSort/results/gen2/arm_a.json`, `results/gen2/arm_a_epochs.csv` and the run folder
`results/gen2/runs/arm_a/`; the summary cell copies the weights to `/kaggle/working/gen2_arm_a_best.pt`.

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/SwarmSort"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Shreyansh303/SwarmSort.git {REPO}
assert os.path.isdir(f"{REPO}/src"), "git clone failed: turn on Settings -> Internet and run again"
%cd {REPO}
# Kaggle's own CUDA build of torch is kept: only Ultralytics is installed, at the version pinned in requirements.txt
!pip install -q ultralytics==8.4.169
import torch, ultralytics
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator -> GPU T4 x2 and run again"
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))
if not os.path.isfile("src/watchdog_run.py"):
    raise RuntimeError("src/watchdog_run.py is not on GitHub yet: push the latest commits")
sys.path.insert(0, f"{REPO}/src")
from watchdog_run import run_watched
# About 10 s: check on this machine that a frozen script and its worker process are killed and retried
subprocess.run([sys.executable, "-m", "unittest", "-q",
                "tests.test_watchdog.TestWatchdog.test_b_stalled_script_is_killed_and_retried",
                "tests.test_watchdog.TestWatchdog.test_e_child_processes_are_killed_too"], check=True)


def run(*args, watch=(), stall_minutes=20, max_retries=3):
    """Run a repository script under the stall watchdog (src/watchdog_run.py). Its output is streamed here. If it
    prints nothing and nothing changes under `watch` for stall_minutes, it is killed together with its dataloader
    workers and started again with the same arguments, at most max_retries times (search.py then continues from its
    log, train_final.py --resume from last.pt). If the script stops with an error, this cell fails and the version
    is marked failed (as with check=True)."""
    args = [str(a) for a in args]
    print("$ python", " ".join(args), flush=True)
    run_watched([sys.executable, *args], watch=watch, stall_minutes=stall_minutes, max_retries=max_retries,
                env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
# Find the dataset built by g2_0_build_dataset (attached as input), copy it to this machine's local disk, and write
# configs that point at the local copy: /kaggle/working/gen2_cfg/data.yaml, data_test_<domain>.yaml, split.csv, lists/
# Why the copy: two Gen 2 runs froze about 55 minutes into training and sat idle until Kaggle's 12-hour limit,
# most likely because the attached notebook-output mount under /kaggle/input stopped answering. Training now reads
# only the local copy. It goes outside /kaggle/working, so it does not become part of this version's output.
# The copy reads that mount too, so it fails loudly (TimeoutError) if no file arrives for 2 minutes.
import glob, json, shutil
from watchdog_run import copy_files, list_tree
INPUT, CFG = "/kaggle/input", "/kaggle/working/gen2_cfg"
builds = sorted({os.path.dirname(p) for p in glob.glob(f"{INPUT}/**/build_report.json", recursive=True)
                 if os.path.isfile(os.path.join(os.path.dirname(p), "data.yaml"))
                 and os.path.isdir(os.path.join(os.path.dirname(p), "images"))})
if not builds:
    raise FileNotFoundError("The Generation 2 dataset is not attached. Add Input -> Notebook Output -> "
                            "g2_0_build_dataset (its finished version), then run again.")
if len(builds) > 1:
    raise RuntimeError("Several built datasets are attached; remove all but one version of g2_0_build_dataset:\n  "
                       + "\n  ".join(builds))
BUILD_SRC = builds[0]
print("Built dataset (attached):", BUILD_SRC)
files = list_tree(BUILD_SRC, exclude=["*.cache"])
size = sum(n for _, n in files)
for base in ("/kaggle/tmp", "/kaggle/temp", "/tmp"):  # local disk outside /kaggle/working
    if os.path.isdir(base) and os.access(base, os.W_OK) and shutil.disk_usage(base).free > 1.1 * size + 2e9:
        BUILD = f"{base}/gen2_data"
        break
else:
    raise OSError(f"No local folder has {size / 1e9:.1f} GB free for the dataset copy")
stamp = {"source": BUILD_SRC, "files": len(files), "bytes": size}
if os.path.isfile(f"{BUILD}/.copied.json") and json.load(open(f"{BUILD}/.copied.json")) == stamp:
    print("Local copy already made in this session:", BUILD)
else:
    shutil.rmtree(BUILD, ignore_errors=True)
    print(f"Copying {len(files)} files ({size / 1e9:.2f} GB) to {BUILD}", flush=True)
    info = copy_files(BUILD_SRC, BUILD, files, workers=16, stall_seconds=120)
    if list_tree(BUILD) != files:
        raise RuntimeError(f"The local copy {BUILD} differs from {BUILD_SRC} (file names or sizes)")
    json.dump(stamp, open(f"{BUILD}/.copied.json", "w"))
    print(f"Copied {info['files']} files, {info['bytes'] / 1e9:.2f} GB in {info['seconds']:.0f} s "
          f"({info['bytes'] / 1e6 / max(info['seconds'], 1):.0f} MB/s); names and sizes match the attached build")
for sub in ("images", "labels"):
    print(f"  {sub}/: {len(os.listdir(f'{BUILD}/{sub}'))} files")
if "--relocate-from" not in open("src/build_dataset.py", encoding="utf-8").read():
    raise RuntimeError("src/build_dataset.py on GitHub has no --relocate-from mode yet: push the latest commits")
run("src/build_dataset.py", "--relocate-from", BUILD, "--out", CFG)
assert os.path.isfile(f"{CFG}/data.yaml"), f"the relocation did not write {CFG}/data.yaml"
print(open(f"{CFG}/data.yaml").read())
print("Test-set yamls:", sorted(os.path.basename(p) for p in glob.glob(f"{CFG}/data_test_*.yaml")))


In [ ]:
# Salvage: if a previous version of this notebook stopped or froze, its output holds results/gen2/runs/arm_a/
# (weights/last.pt, results.csv). Attach that version (Add Input -> Notebook Output -> this notebook) and its run
# folder is copied here, so the training below continues from its last.pt (train_final.py --resume) instead of
# starting again. With several attached versions, the one with the most epochs in results.csv is used.
import csv, yaml
ARM_RUN = "results/gen2/runs/arm_a"
if os.path.exists(f"{ARM_RUN}/weights/last.pt"):
    print(ARM_RUN, "already present: the training continues from it")
else:
    found = []
    for last in glob.glob(f"{INPUT}/**/{ARM_RUN}/weights/last.pt", recursive=True):
        previous = os.path.dirname(os.path.dirname(last))
        try:
            saved = yaml.safe_load(open(f"{previous}/args.yaml"))
            with open(f"{previous}/results.csv", newline="") as f:
                epochs_done = len(list(csv.DictReader(f)))
        except Exception as e:
            print(f"Skipping {previous}: unreadable ({type(e).__name__}: {e})")
            continue
        if saved.get("epochs") != 100 or saved.get("imgsz") != 640:
            print(f"Skipping {previous}: not the 100-epoch training at imgsz 640")
            continue
        found.append((epochs_done, previous))
    if found:
        epochs_done, previous = max(found)
        copy_files(previous, ARM_RUN, list_tree(previous), workers=4, stall_seconds=120)
        print(f"Continuing from {previous} (its results.csv has {epochs_done} of 100 epochs): train_final.py "
              f"--resume continues its last.pt after checking that its settings are this notebook's")
    else:
        print(f"No previous {ARM_RUN} attached: the training starts at epoch 1")


In [ ]:
# Full training with the Ultralytics defaults (no --config) on one GPU (device 0). --resume continues
# results/gen2/runs/arm_a/weights/last.pt if the salvage cell or a watchdog retry left one; otherwise a normal run.
run("src/train_final.py", "--data", f"{CFG}/data.yaml", "--imgsz", 640, "--name", "arm_a",
    "--out", "results/gen2/arm_a.json", "--project", "results/gen2/runs", "--device", 0, "--resume",
    watch=["results/gen2/runs/arm_a"])


In [ ]:
# Val metrics of best.pt, a check that the defaults were trained with the full recipe,
# and a copy of the weights at the top of /kaggle/working so they are easy to find in the Output tab
import json, math, shutil
sys.path.insert(0, "src")
from search_space import DEFAULTS, NAMES
ARM = "arm_a"
res = json.load(open(f"results/gen2/{ARM}.json"))
wrong = [k for k in NAMES if not math.isclose(res["params"][k], DEFAULTS[k], rel_tol=1e-9)]
assert not wrong, f"results/gen2/{ARM}.json was not trained with the Ultralytics defaults for {wrong}"
assert res["epochs"] == 100 and res["imgsz"] == 640 and not res["smoke"], "not the full 100-epoch recipe at imgsz 640"
print("Trained with the Ultralytics defaults:", ", ".join(f"{k}={res['params'][k]:g}" for k in NAMES))
print(f"\nval split, best.pt")
for key, label in [("map50", "mAP@50"), ("map50_95", "mAP@50-95"), ("precision", "precision"), ("recall", "recall")]:
    print(f"  {label:<12}{res[key]:.4f}")
print(f"  training time {res['seconds'] / 3600:.2f} h on {res['device']} (estimate: 2.4 h)")
print("Val numbers only: the fair comparison is the test split, evaluated once in g2_6_evaluate.")

weights = f"results/gen2/runs/{ARM}/weights/best.pt"
assert os.path.exists(weights), f"{weights} not found (see the output of the training cell)"
shutil.copyfile(weights, f"/kaggle/working/gen2_{ARM}_best.pt")
print(f"\nCopied {weights} -> /kaggle/working/gen2_{ARM}_best.pt ({os.path.getsize(weights) / 1e6:.1f} MB)")


### Files to download / where they go locally

| File in the Output tab | Where it goes locally |
|---|---|
| `SwarmSort/results/gen2/arm_a.json` | `results/gen2/arm_a.json` |
| `SwarmSort/results/gen2/arm_a_epochs.csv` | `results/gen2/arm_a_epochs.csv` |
| `gen2_arm_a_best.pt` (top level of the output) | `results/gen2/runs/arm_a/weights/best.pt` (create the folders) |

**The `.pt` file downloads as a `.zip`**, because a PyTorch checkpoint is itself a zip archive. Do not unzip it: just
rename the downloaded file to `best.pt`. Commit the JSON and CSV; the weights stay local (`runs/` and `*.pt` are
git-ignored). `g2_6_evaluate` takes the weights straight from this notebook's output, so attach this version there.

### If the session dies or freezes

**If a version stopped or froze, attach its output and run again: it continues.** Add Input -> Notebook Output ->
this notebook, pick the stopped version (keep `g2_0_build_dataset` attached), and Save & Run All. The salvage cell
copies its `results/gen2/runs/arm_a/` here and `train_final.py --resume` continues from its `last.pt` (Ultralytics
resume: same weights, optimizer state and epoch counter), so only the unfinished epochs are trained. Without an
attached run folder the training starts again from epoch 1 (seeded and deterministic). Within one session the
watchdog restarts a frozen training by itself, also with `--resume`.